# 06 - Curve-level and per-pixel metrics on known inputs

This notebook confirms the aggregate metrics computed by `Metrics.compute`. We build a synthetic `Result` whose prediction equals the target plus a controlled perturbation, so the curve-level MSE / MAE / RMSE / R-squared, per-pixel statistics, and SSIM summaries can be sanity-checked against hand computations.

Modules exercised: `pipelines.inference_pipeline.metrics.Metrics` and the `Result` dataclass it consumes.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Build a synthetic Result

We make a small cube of shape `(n_elev, H, W)`. The GT is a smooth field, the prediction adds Gaussian noise of known standard deviation. The per-pixel metric maps required by `Result` are filled with derived values; pixel R-squared, cosine and peak error use simple placeholders since this notebook targets the curve-level path.

In [ ]:
from pipelines.inference_pipeline.metrics import Metrics, Result

n_elev, H, W = 16, 8, 8
x_axis = np.linspace(-10.0, 10.0, n_elev).astype(np.float64)
rng    = np.random.default_rng(0)

base = np.exp(-((x_axis[:, None, None]) ** 2) / 8.0).astype(np.float32)
gt   = base + 0.1 * rng.random((n_elev, H, W)).astype(np.float32)

noise_std = 0.05
pred = gt + noise_std * rng.standard_normal((n_elev, H, W)).astype(np.float32)

pixel_mse  = ((pred - gt) ** 2).mean(axis=0)
pixel_mae  = np.abs(pred - gt).mean(axis=0)
result = Result(
    pred_curves=pred, gt_curves=gt,
    params_pred=rng.random((6, H, W)).astype(np.float32),
    params_gt=rng.random((6, H, W)).astype(np.float32),
    pixel_mse=pixel_mse.astype(np.float32),
    pixel_mae=pixel_mae.astype(np.float32),
    pixel_r2=np.full((H, W), 0.9, np.float32),
    pixel_cosine=np.full((H, W), 0.99, np.float32),
    pixel_peak_err_idx=np.zeros((H, W), np.int32),
    cube_directory=Path('.'), azimuth_offset=0, range_offset=0,
)
print('Result built with cube shape', pred.shape)


## Compute the metrics

`Metrics.compute` runs the full aggregation, including a multiprocess SSIM over the slice indices we pass. We request all indices on each axis.

In [ ]:
metrics = Metrics(result, x_axis, n_gaussians=2)
global_metrics = metrics.compute(
    elev_indices=np.arange(n_elev),
    range_indices=np.arange(W),
    az_indices=np.arange(H),
)
print('total metric keys:', len(global_metrics))


## Cross-check curve-level scalars by hand

We recompute MSE, MAE, RMSE and global R-squared directly from the arrays and compare to the reported values; they must agree to floating-point tolerance.

In [ ]:
diff   = pred.astype(np.float64) - gt.astype(np.float64)
mse_ref  = float((diff ** 2).mean())
mae_ref  = float(np.abs(diff).mean())
rmse_ref = float(np.sqrt(mse_ref))
gt_mean  = float(gt.mean(dtype=np.float64))
r2_ref   = 1.0 - float((diff ** 2).sum()) / (float(((gt - gt_mean) ** 2).sum()) + 1e-12)

rows = [
    ('MSE',  mse_ref,  global_metrics['curve_mse_gt']),
    ('MAE',  mae_ref,  global_metrics['curve_mae_gt']),
    ('RMSE', rmse_ref, global_metrics['curve_rmse_gt']),
    ('R^2',  r2_ref,   global_metrics['overall_r2_gt']),
]
for name, ref, got in rows:
    print(f'{name:5s} hand={ref:.6f}  metrics={got:.6f}  match={abs(ref-got) < 1e-5}')


## Visualise headline scalars and per-pixel distributions

A bar chart of the curve-level scalars plus histograms of the per-pixel MSE and MAE maps give a quick visual read of error magnitude and spread.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.6))

scalar_names = ['curve_mse_gt', 'curve_mae_gt', 'curve_rmse_gt', 'overall_r2_gt']
scalar_vals  = [global_metrics[k] for k in scalar_names]
axes[0].bar(range(len(scalar_names)), scalar_vals, color='C0', alpha=0.8)
axes[0].set_xticks(range(len(scalar_names)))
axes[0].set_xticklabels(['MSE', 'MAE', 'RMSE', 'R2'])
axes[0].set_title('Curve-level scalars')

axes[1].hist(pixel_mse.reshape(-1), bins=20, color='C1', alpha=0.8)
axes[1].set_title('per-pixel MSE'); axes[1].set_xlabel('MSE')

axes[2].hist(pixel_mae.reshape(-1), bins=20, color='C2', alpha=0.8)
axes[2].set_title('per-pixel MAE'); axes[2].set_xlabel('MAE')

fig.tight_layout()
plt.show()


## SSIM summaries are present and bounded

The compute call also produces mean SSIM per axis. The reported values must be finite and bounded within `[-1, 1]`. Their magnitude reflects how much structure the GT carries: here the GT is a smooth profile plus a sizeable random component, so SSIM is moderate rather than near one.

In [ ]:
for key in ['ssim_gt_elev_mean', 'ssim_gt_range_mean', 'ssim_gt_azimuth_mean']:
    val = global_metrics[key]
    print(f'{key:24s}: {val:.4f}  bounded={-1.0 <= val <= 1.0}')


## Expected visual outcome

The hand-computed MSE, MAE, RMSE and R-squared must match the reported values within `1e-5`. The curve-level RMSE should be near the injected noise level (slightly above `0.05`), the per-pixel histograms should be tightly clustered at small errors, and all three mean SSIM values must be finite and bounded in `[-1, 1]`.